# 4주차 직접 해보기: 어텐션과 트랜스포머

사이트의 4주차 회독과 정리 슬라이드를 본 다음 풀어요. 위에서부터 차례로 `Shift+Enter` 로 실행해요.

- 문제마다 **생각 순서**가 먼저 나와요. 코드를 치기 전에 그 순서를 말로 한 번 읊어요.
- `# TODO` 가 있는 칸의 `None` 을 알맞은 코드로 바꿔요.
- 바로 아래 **확인 셀**을 실행하면 맞았는지 알려 줘요. 틀리면 빨간 AssertionError 와 힌트가 나와요.
- 막히면 맨 아래 **정답 코드**를 봐요.

GPU 는 필요 없어요. 인터넷 다운로드도 없어요.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
torch.manual_seed(0)
def ok(cond, msg):
    assert cond, msg
    print('정답! ' + msg)
print('준비 끝')

## 1. 1단계: 내적으로 어텐션 점수 만들기

강의 N4 p.21. 지금 디코더 상태 s 를 인코더 상태 h 하나하나와 **내적(Dot Product)** 해서 점수를 만들어요. 점수가 크면 그 자리가 지금 중요하다는 뜻이에요.

`H @ s` 는 H 의 각 줄과 s 를 내적한 결과를 한 번에 줘요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 점수는 몇 개 나와야 하나요? 인코더 상태 개수만큼, 곧 4개
2. 점수 하나 = s 와 h 를 같은 자리끼리 곱해서 더하기
3. for 문으로 쓰면: for h in H: score = sum(s[i] * h[i] for i in ...)
4. 행렬로 쓰면 그 for 문이 H @ s 한 줄이 돼요
5. 모양 확인: (4, 2) @ (2,) -> (4,)

In [ ]:
s = np.array([1.0, 2.0])
H = np.array([[3.0, 0.3],
              [0.4, 0.2],
              [0.1, 0.2],
              [0.2, 0.1]])

scores = None   # TODO: H 의 각 줄과 s 의 내적

print(scores, scores.shape)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(scores.shape == (4,), '점수는 인코더 상태 개수만큼 4개')
ok(np.allclose(scores, [3.6, 0.8, 0.5, 0.4]), '점수 [3.6, 0.8, 0.5, 0.4]')
ok(int(np.argmax(scores)) == 0, '첫 자리가 가장 관련이 커요')

## 2. 2단계: 소프트맥스를 직접 만들기

강의 N4 p.22. 점수는 합이 1 이 아니라서 그대로 가중치로 못 써요. **소프트맥스(Softmax)** 로 합이 1 인 분포를 만들어요.

큰 수에 `exp` 를 씌우면 값이 넘칠 수 있어서, 먼저 최댓값을 빼 주는 게 안전한 구현이에요. 빼도 결과는 똑같아요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 최댓값을 빼요. 결과는 안 변하고 넘침만 막아요
2. 각 값에 exp 를 씌워요
3. 전체 합으로 나눠요
4. 말로 하면 빼기, exp, 합으로 나누기 세 줄이에요
5. 확인: 결과를 다 더하면 정확히 1 이어야 해요

In [ ]:
def softmax(x):
    x = x - x.max()            # 넘침 막기
    e = None                   # TODO: exp 씌우기
    return None                # TODO: 전체 합으로 나누기

alpha = softmax(np.array([2.0, 1.0, 0.0]))
print(alpha, alpha.sum())

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(alpha.sum() - 1.0) < 1e-12, '다 더하면 1')
ok(np.allclose(alpha, [0.6652, 0.2447, 0.0900], atol=1e-4), '[0.6652, 0.2447, 0.0900]')
ok(np.allclose(softmax(np.array([2.0, 1.0, 0.0]) + 100), alpha), '전체에 상수를 더해도 결과가 같아요')

## 3. 3단계: 가중합으로 문맥 벡터 만들기

강의 N4 p.23. 가중치를 각 상태에 곱해서 모두 더하면 **문맥 벡터(Context Vector)** 가 나와요. 가중치 합이 1 이므로 가중 평균이에요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 문맥 벡터의 칸 수는 상태 하나와 같아요, 곧 2칸
2. for 문으로 쓰면 c = alpha[0]*h0 + alpha[1]*h1 + alpha[2]*h2
3. 행렬로 쓰면 (3,) @ (3, 2) -> (2,)
4. 가중치가 큰 자리의 내용이 많이 들어갔는지 눈으로 확인해요

In [ ]:
alpha3 = np.array([0.5, 0.3, 0.2])
Hs = np.array([[2.0, 0.0],
               [0.0, 2.0],
               [1.0, 1.0]])

context = None   # TODO: 가중합

print(context)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(context.shape == (2,), '문맥 벡터는 상태 하나와 같은 2칸')
ok(np.allclose(context, [1.2, 0.8]), '[1.2, 0.8]')
ok(np.allclose(np.array([0.5, 0.3, 0.2]) @ Hs, context), '가중치 합이 1 이라 가중 평균이에요')

## 4. 왜 루트 d 로 나눌까

강의 N4 p.38. 점수가 커질수록 소프트맥스가 뾰족해져요. 뾰족해지면 **기울기(Gradient)** 가 거의 0 이 돼서 학습이 멈춰요. 그래서 점수를 $\sqrt{d_k}$ 로 나눠요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 같은 점수로 두 번 계산해 보면 돼요
2. 그냥 소프트맥스를 하고 최댓값을 봐요
3. sqrt(d_k) 로 나눈 뒤 소프트맥스를 하고 최댓값을 봐요
4. d_k = 4 이니까 나누는 값은 2 예요
5. 어느 쪽이 더 평평해졌는지 숫자로 비교해요

In [ ]:
raw = np.array([4.0, 2.0, 0.0])
d_k = 4

unscaled = softmax(raw)
scaled = None   # TODO: raw 를 math.sqrt(d_k) 로 나눈 뒤 softmax

print(unscaled.max(), scaled.max())

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(unscaled.max() - 0.8668) < 1e-3, '나누기 전 최대 가중치 약 0.8668')
ok(abs(scaled.max() - 0.6652) < 1e-3, '나눈 뒤 최대 가중치 약 0.6652')
ok(scaled.max() < unscaled.max(), '나누면 분포가 덜 뾰족해져요')

## 5. 한 입력에서 Q, K, V 만들기

강의 N4 p.34. 토큰 하나가 **쿼리(Query)**, **키(Key)**, **밸류(Value)** 세 역할을 동시에 해요. 같은 입력 X 에 서로 다른 행렬 세 개를 곱해서 만들어요. 세 행렬은 모든 자리가 함께 써요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. X 의 모양은 (토큰 수, 입력 칸 수) = (3, 4) 예요
2. W 의 모양은 (입력 칸 수, 결과 칸 수) = (4, 2) 예요
3. (3, 4) @ (4, 2) -> (3, 2). 가운데 4 가 맞아야 곱할 수 있어요
4. Q, K, V 세 번 똑같이 하면 끝이에요
5. 여기서는 K 를 만드는 행렬이 Q 와 같게 준비돼 있어요

In [ ]:
X  = np.array([[1.0, 0.0, 1.0, 0.0],
               [0.0, 1.0, 0.0, 1.0],
               [1.0, 1.0, 0.0, 0.0]])
Wq = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
Wk = Wq.copy()
Wv = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])

Q = None   # TODO
K = None   # TODO
V = None   # TODO

print(Q.shape, K.shape, V.shape)
print(Q)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(Q.shape == (3, 2) and K.shape == (3, 2) and V.shape == (3, 2), '셋 다 (3, 2)')
ok(np.allclose(Q, [[2, 0], [0, 2], [1, 1]]), 'Q = [[2,0],[0,2],[1,1]]')
ok(np.allclose(V, [[1, 2], [1, 0], [2, 1]]), 'V = [[1,2],[1,0],[2,1]]')

## 6. 셀프 어텐션을 행렬 한 번에

강의 N4 p.36-37. $\mathrm{Attention}(Q,K,V) = \mathrm{softmax}(QK^{\top}/\sqrt{d_k})V$. 자리를 도는 for 문이 없어서 GPU 가 아주 잘해요.

줄마다 소프트맥스를 해야 하므로 `axis=1` 로 합이 1 이 되게 만들어요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 먼저 모양을 종이에 적어요. Q 는 (3,2), K 전치는 (2,3), 점수는 (3,3)
2. 점수를 sqrt(d_k) 로 나눠요. 여기서 d_k 는 Q 의 칸 수 2 예요
3. 줄마다 소프트맥스를 하면 A 는 (3,3) 이고 줄 합이 각각 1 이에요
4. A @ V 는 (3,3) @ (3,2) = (3,2) 예요
5. 마지막 줄은 점수가 모두 같아서 가중치가 3분의 1 씩 나올 거예요

In [ ]:
def row_softmax(M):
    M = M - M.max(axis=1, keepdims=True)
    e = np.exp(M)
    return e / e.sum(axis=1, keepdims=True)

d_k = Q.shape[1]
S = None   # TODO: Q 와 K 전치를 곱하고 math.sqrt(d_k) 로 나누기 (K.T 사용)
A = None   # TODO: 줄마다 소프트맥스
O = None   # TODO: 가중치로 V 를 섞기

print(A.round(4))
print(O.round(4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(S.shape == (3, 3) and A.shape == (3, 3) and O.shape == (3, 2), '점수와 A 는 (3,3), 출력은 (3,2)')
ok(np.allclose(A.sum(axis=1), 1.0), '줄마다 합이 1')
ok(np.allclose(A[0], [0.7679, 0.0454, 0.1867], atol=1e-4), '첫 줄 [0.7679, 0.0454, 0.1867]')
ok(np.allclose(A[2], [1/3, 1/3, 1/3]), '셋째 줄은 점수가 모두 같아 3분의 1 씩')

## 7. 배리어 3: 미래 가리기

강의 N4 p.43. **언어 모델(Language Model (LM))** 은 아직 만들지 않은 토큰을 보면 안 돼요. 미래 자리의 점수를 **소프트맥스 전에** 마이너스 무한으로 바꿔요. `exp(-inf)` 가 0 이라 가중치가 정확히 0 이 돼요.

`np.triu(np.ones((n, n)), 1)` 는 대각선 위쪽만 1 인 표를 만들어요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 가려야 하는 칸은 j 가 i 보다 큰 곳, 곧 오른쪽 위 삼각형이에요
2. np.triu(..., 1) 로 그 자리를 찾아요
3. 그 자리 점수를 -np.inf 로 바꿔요. 0 으로 바꾸면 안 돼요
4. 그다음에 줄마다 소프트맥스를 해요
5. 확인: 첫 줄은 [1, 0, 0] 이고 모든 줄 합이 1 이에요

In [ ]:
raw_scores = np.array([[2.0, 1.0, 0.0],
                       [1.0, 2.0, 1.0],
                       [0.0, 1.0, 2.0]])

future = np.triu(np.ones((3, 3)), 1) == 1
masked = None   # TODO: future 자리를 -np.inf 로 (np.where 사용)
Am = None       # TODO: 줄마다 소프트맥스

print(Am.round(4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(np.allclose(Am.sum(axis=1), 1.0), '마스크를 씌워도 줄마다 합은 정확히 1')
ok(np.allclose(Am[0], [1.0, 0.0, 0.0]), '첫 줄은 자기 자신만 봐요')
ok(np.allclose(Am[1], [0.2689, 0.7311, 0.0], atol=1e-4), '둘째 줄 [0.2689, 0.7311, 0]')
ok(np.allclose(np.triu(Am, 1), 0.0), '오른쪽 위가 전부 정확히 0')

## 8. 멀티 헤드: 나누고, 보고, 이어 붙이기

강의 N4 p.47-48. 전체 차원 d 를 헤드 h 개로 나눠서 d/h 차원에서 각각 어텐션을 해요. 결과를 이어 붙이고 $W^O$ 로 섞어요.

핵심은 `reshape` 으로 (배치, 토큰, d) 를 (배치, 헤드, 토큰, d/h) 로 바꾸는 것이에요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. d = 8, h = 2 이니까 헤드 하나의 차원은 8 나누기 2 로 4 예요
2. (1, 3, 8) 을 (1, 3, 2, 4) 로 reshape 하면 헤드가 갈라져요
3. 그다음 transpose 로 (1, 2, 3, 4), 곧 (배치, 헤드, 토큰, 칸) 으로 바꿔요
4. 파라미터 수는 Linear(8, 8) 하나가 가중치 64 더하기 편향 8 로 72 개예요
5. Q, K, V, O 네 개면 72 곱하기 4 로 288 개. 헤드를 몇 개로 나눠도 이 수는 그대로예요

In [ ]:
d, h, n = 8, 2, 3
head_dim = None   # TODO: d 를 h 로 나눈 값 (//)

x = torch.arange(float(n * d)).reshape(1, n, d)
split = x.reshape(1, n, h, head_dim).transpose(1, 2)   # (배치, 헤드, 토큰, 칸)

Wq, Wk, Wv, Wo = (nn.Linear(d, d) for _ in range(4))
n_params = None   # TODO: 네 Linear 의 파라미터 수를 모두 더하기 (p.numel() 사용)

print(head_dim, tuple(split.shape), n_params)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(head_dim == 4, '헤드 하나의 차원은 d 나누기 h 로 4')
ok(tuple(split.shape) == (1, 2, 3, 4), '(배치, 헤드, 토큰, 칸) = (1, 2, 3, 4)')
ok(n_params == 288, '네 Linear 합쳐 288개. 헤드 수와 상관없어요')
ok(torch.allclose(split.transpose(1, 2).reshape(1, n, d), x), '되돌리면 원래 x 와 같아요. 이게 이어 붙이기예요')

## 9. 사인 코사인 위치 인코딩 만들기

강의 N4 p.40. $PE_{pos,2i} = \sin(pos/10000^{2i/d})$, $PE_{pos,2i+1} = \cos(pos/10000^{2i/d})$.

짝수 칸은 sin, 홀수 칸은 cos 이에요. 학습하는 값이 아니라 계산하는 값이라 **파라미터(Parameter)** 가 0 개예요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 만들 표의 모양은 (자리 수, d) = (3, 4) 예요
2. 칸 쌍마다 각도가 달라요. 각도는 pos 나누기 10000의 (2i/d) 제곱이에요
3. 짝수 칸 [:, 0::2] 에 sin, 홀수 칸 [:, 1::2] 에 cos 을 넣어요
4. 자리 0 은 sin(0)=0, cos(0)=1 이라 (0, 1, 0, 1) 이 나와야 해요
5. 두 자리의 벡터가 서로 다른지 확인하면 순서가 살아난 거예요

In [ ]:
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = None   # TODO: sin
    pe[:, 1::2] = None   # TODO: cos
    return pe

P = sinusoidal_pe(3, 4)
print(P)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(tuple(P.shape) == (3, 4), '(자리 3개, 칸 4개)')
ok(torch.allclose(P[0], torch.tensor([0.0, 1.0, 0.0, 1.0]), atol=1e-5), '자리 0 은 (0, 1, 0, 1)')
ok(torch.allclose(P[1][:2], torch.tensor([0.8415, 0.5403]), atol=1e-3), '자리 1 의 앞 두 칸 (0.8415, 0.5403)')
ok(not torch.allclose(P[1], P[2]), '자리마다 값이 달라서 순서를 구별할 수 있어요')

## 10. 층 정규화와 Pre-LN 블록

강의 N4 p.50-51. **층 정규화(Layer Normalization (LayerNorm))** 는 한 토큰의 숫자들을 평균 0, 분산 1 로 맞춰요. 토큰끼리도 배치끼리도 섞지 않아요.

Pre-LN 블록은 `x = x + Attention(LN(x))` 다음에 `x = x + FFN(LN(x))` 예요. 이 더하기가 **잔차 연결(Residual Connection)** 이에요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 먼저 손으로 해요. x = (2, 4, 4, 10) 의 평균은 20 나누기 4 로 5 예요
2. 편차는 (-3, -1, -1, 5), 제곱 평균인 분산은 36 나누기 4 로 9, 표준편차는 3 이에요
3. 정규화 결과는 (-1, -1/3, -1/3, 5/3) 이에요
4. 블록은 더하기 두 번이에요. x + f(LN(x)) 를 두 번 해요
5. 모양은 절대 바뀌지 않아요. 들어간 모양 그대로 나와요

In [ ]:
x = torch.tensor([2.0, 4.0, 4.0, 10.0])
ln = nn.LayerNorm(4, elementwise_affine=False)

by_hand = None   # TODO: (x - 평균) / 표준편차. x.mean(), x.std(unbiased=False) 사용

d = 4
block_ln1, block_ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))
attn = nn.Linear(d, d)   # 어텐션 자리에 넣은 가짜 층

h = x.reshape(1, 1, d)
h = None   # TODO: h + attn(block_ln1(h))
h = h + ffn(block_ln2(h))

print(by_hand, tuple(h.shape))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(torch.allclose(by_hand, torch.tensor([-1.0, -1/3, -1/3, 5/3]), atol=1e-5), '손계산 (-1, -0.3333, -0.3333, 1.6667)')
ok(torch.allclose(by_hand, ln(x), atol=1e-4), 'PyTorch LayerNorm 과 같은 값')
ok(abs(float(by_hand.mean())) < 1e-5 and abs(float((by_hand ** 2).mean()) - 1.0) < 1e-4, '평균 0, 분산 1')
ok(tuple(h.shape) == (1, 1, 4), '블록을 지나도 모양은 그대로 (1, 1, 4)')

## 정답 코드

먼저 스스로 풀어 보고, 막혔을 때만 봐요.

**1. 1단계: 내적으로 어텐션 점수 만들기**

```python
s = np.array([1.0, 2.0])
H = np.array([[3.0, 0.3],
              [0.4, 0.2],
              [0.1, 0.2],
              [0.2, 0.1]])

scores = H @ s

print(scores, scores.shape)
```

**2. 2단계: 소프트맥스를 직접 만들기**

```python
def softmax(x):
    x = x - x.max()            # 넘침 막기
    e = np.exp(x)
    return e / e.sum()

alpha = softmax(np.array([2.0, 1.0, 0.0]))
print(alpha, alpha.sum())
```

**3. 3단계: 가중합으로 문맥 벡터 만들기**

```python
alpha3 = np.array([0.5, 0.3, 0.2])
Hs = np.array([[2.0, 0.0],
               [0.0, 2.0],
               [1.0, 1.0]])

context = alpha3 @ Hs

print(context)
```

**4. 왜 루트 d 로 나눌까**

```python
raw = np.array([4.0, 2.0, 0.0])
d_k = 4

unscaled = softmax(raw)
scaled = softmax(raw / math.sqrt(d_k))

print(unscaled.max(), scaled.max())
```

**5. 한 입력에서 Q, K, V 만들기**

```python
X  = np.array([[1.0, 0.0, 1.0, 0.0],
               [0.0, 1.0, 0.0, 1.0],
               [1.0, 1.0, 0.0, 0.0]])
Wq = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
Wk = Wq.copy()
Wv = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])

Q = X @ Wq
K = X @ Wk
V = X @ Wv

print(Q.shape, K.shape, V.shape)
print(Q)
```

**6. 셀프 어텐션을 행렬 한 번에**

```python
def row_softmax(M):
    M = M - M.max(axis=1, keepdims=True)
    e = np.exp(M)
    return e / e.sum(axis=1, keepdims=True)

d_k = Q.shape[1]
S = Q @ K.T / math.sqrt(d_k)
A = row_softmax(S)
O = A @ V

print(A.round(4))
print(O.round(4))
```

**7. 배리어 3: 미래 가리기**

```python
raw_scores = np.array([[2.0, 1.0, 0.0],
                       [1.0, 2.0, 1.0],
                       [0.0, 1.0, 2.0]])

future = np.triu(np.ones((3, 3)), 1) == 1
masked = np.where(future, -np.inf, raw_scores)
Am = row_softmax(masked)

print(Am.round(4))
```

**8. 멀티 헤드: 나누고, 보고, 이어 붙이기**

```python
d, h, n = 8, 2, 3
head_dim = d // h

x = torch.arange(float(n * d)).reshape(1, n, d)
split = x.reshape(1, n, h, head_dim).transpose(1, 2)   # (배치, 헤드, 토큰, 칸)

Wq, Wk, Wv, Wo = (nn.Linear(d, d) for _ in range(4))
n_params = sum(p.numel() for L in (Wq, Wk, Wv, Wo) for p in L.parameters())

print(head_dim, tuple(split.shape), n_params)
```

**9. 사인 코사인 위치 인코딩 만들기**

```python
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

P = sinusoidal_pe(3, 4)
print(P)
```

**10. 층 정규화와 Pre-LN 블록**

```python
x = torch.tensor([2.0, 4.0, 4.0, 10.0])
ln = nn.LayerNorm(4, elementwise_affine=False)

by_hand = (x - x.mean()) / x.std(unbiased=False)

d = 4
block_ln1, block_ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))
attn = nn.Linear(d, d)   # 어텐션 자리에 넣은 가짜 층

h = x.reshape(1, 1, d)
h = h + attn(block_ln1(h))
h = h + ffn(block_ln2(h))

print(by_hand, tuple(h.shape))
```